

<img src='images/gdd-logo.png' width='300px' align='right' style="padding: 15px">

# Decorators

A decorator is a **design pattern** in Python that allows a user to **add new functionality** to an existing object **without modifying its structure**. For example, easily adding the functionality to time how long a function takes to run, without having to rewrite it. 

They are usually called before the definition of a function you want to decorate. 

Functions in Python are ***first class citizens***. This means they support operations such as being:
- Passed as an argument
- Returned from another function
- Modified
- Assigned to a variable 

A function being a ***first class citizen*** is a fundamental concept to understand to fully understand Python decorators. 

Therefore, notebook is split into two sections:
<img src='images/python-paint.png' width='200px' align='right' style="padding: 15px">

**Decorator Pre-requisites**
- [Assigning Functions to Variables](#assigning)
- [Defining Functions Inside other Functions](#defining)
- [Passing Functions as Arguments to other Functions](#passing)
- [Functions Returning other Functions](#return)
- [Nested Functions have access to the Enclosing Function's Variable Scope](#nested)

**Creating Decorators**
- [Motivation for Decorators](#motivate)
- [Creating your first Decorator](#create)
- [Applying Multiple Decorators](#applying)
- [Accepting Arguments in Decorator Functions](#accepting)
- [Defining General Purpose Decorators](#defining)

[**Usecases of decorators**](#usecases)

<a id='assigning'></a>
## Decorator Pre-requisites

### Assigning Functions to Variables

Let's take a function, that will multiply a number by 2 and assign it to a variable. You can then use this variable to call the function.

In [37]:
def times_two(number):
    return number * 2

multiply_two = times_two
multiply_two(5)

10

<a id='defining'></a>
### Defining Functions Inside other Functions
Next, let's define a function inside another function in Python. This becomes very relevant when creating and understanding decorators.

In [38]:
def multiply_two(number):
    def times_two(number):
        return number * 2

    result = times_two(number)
    return result

multiply_two(11)

22

<a id='passing'></a>
### Passing Functions as Arguments to other Functions
Functions can also be passed as arguments to other functions. 

In [39]:
def multiply_two(number):
    return number * 2

def function_call(function):
    number_to_multiply = 5
    return function(number_to_multiply)

function_call(multiply_two)

10

<a id='return'></a>
### Functions Returning other Functions
A function can also generate another function. 

In [40]:
def hello_function():
    
    def say_hi():
        return "Hello!"
    
    return say_hi

hello = hello_function()
hello()

'Hello!'

<a id='nested'></a>
### Nested Functions have access to the Enclosing Function's Variable Scope
Python allows a nested function to access the outer scope of the enclosing function. This is a critical concept in decorators -- this pattern is known as ***Closure***.

In [41]:
def hello_function(name='World'):
    "Enclosing Function"
    
    def say_hi():
        "Nested Function"
        print(f"Hello, {name}!")
    
    say_hi()

hello_function()

Hello, World!


---
<a id='motivate'></a>
## Creating Decorators
### Motivation

Okay, now that you revisited the operations that you can perform on functions, lets look at how this ties in with Python Decorators:

Below is a simple function that returns some text.

In [42]:
def say_hi():
    return 'hello, world!'
say_hi()

'hello, world!'

Let's imagine you wanted the option of returning the text in upper case. 

Of course, one option would be to simply rewrite our function. 

In [43]:
def say_hi(upper=False):
    if upper:
        return 'hello, world!'.upper()
    else:
        return 'hello, world!'
say_hi(upper=True)

'HELLO, WORLD!'

Alternatively, you could add this functionality by creating a decorator! This has the benefit of leaving our original function unchanged. Furthermore, it would allow you to add the functionality to other functions, too!

<a id='create'></a>
### Creating your first Decorator

You will now create a simple decorator that can convert the output of a function to uppercase.

How do you do this?
 1. The decorator function takes a **function as an argument.**
 2. You then create a **wrapper function**, which wraps around the function you passed and modifies its inputs or outputs.
 3. Lastly, the decorator function **returns the wrapper function**.

In [44]:
def uppercase_decorator(function):
    
    def wrapper():
        return function().upper()

    return wrapper

Notice that the decorator takes a function as an argument and returns a different function!

To modify your function, you could assign the decorator function that uses the say_hi as input to a variable that you could then call later on.

In [45]:
def say_hi():
    return 'hello, world!'

decorate = uppercase_decorator(say_hi)
decorate()

'HELLO, WORLD!'

However, Python provides a much easier way for you to apply decorators. You simply use the `@` symbol before the function you would like to decorate. Let's show that in practice below.

In [46]:
@uppercase_decorator
def say_hi():
    return 'hello, world!'

say_hi()

'HELLO, WORLD!'

### <mark>Exercise: Create a "Split String" decorator</mark>

Create a decorator that would change a string into a list of words and test it on the `say_hi` function.

In [47]:
def split_string_decorator(function):
    
    def wrapper():
        return function().split()

    return wrapper

In [48]:
@split_string_decorator
def say_hi():
    return 'hello, world!'

say_hi()

['hello,', 'world!']

In [49]:
# %load answers/ex-decorators1.py

<mark>**Bonus:**</mark> Create a decorator named `time_it` that prints the number of seconds a function needed to be executed.

*Hint: Import the `time` package from base Python and use `time.time()` before and after the function execution to calculate a time difference.*

In [50]:
import time 

def time_it_decorator(function):
    
    def wrapper():
        start_time = time.time()
        result = function()
        end_time = time.time()
        print(f'The function {function.__name__} took {round(end_time-start_time, 4)} seconds to run.')
        return result

    return wrapper

In [51]:
@time_it_decorator
def say_hi():
    for i in range(10): 
        print(f"Hello, world! {i}")
    return

say_hi()

Hello, world! 0
Hello, world! 1
Hello, world! 2
Hello, world! 3
Hello, world! 4
Hello, world! 5
Hello, world! 6
Hello, world! 7
Hello, world! 8
Hello, world! 9
The function say_hi took 0.0001 seconds to run.


In [52]:
# %load answers/ex-decorators1-bonus.py
import time

def time_it(function):
    def wrapper():
        t1 = time.time()
        result = function()
        t2 = time.time()
        print(f'The function {function.__name__} took {round(t2-t1, 4)} seconds to run.')
        return result
    return wrapper

@time_it
def say_hi():
    return 'hello, world'

say_hi()


The function say_hi took 0.0 seconds to run.


'hello, world'

<a id='applying'></a>
### Applying Multiple Decorators
You can also apply multiple decorators to a single function. 

Let's apply both of our decorators to the `say_hi` function being mindful about order.

In [53]:
@time_it_decorator
@split_string_decorator
@uppercase_decorator
def say_hi():
    return 'hello, world'

say_hi()

The function wrapper took 0.0 seconds to run.


['HELLO,', 'WORLD']

<mark>***Questions:***</mark>

1. In what order do the decorators get applied? (top down or bottom up?)

2. Why can't you reverse the order of these decorators?

<details>
    
  <summary><span style="color:blue">Show answers</span></summary>
  
1. First, the `uppercase_decorator` is applied and then the `split_string` decorator.
2. Because the `uppercase_decorator` applies the `.upper()` method which does not work on lists (which you would have, if you applied `split_string` first). 
    
</details>

<a id='accepting'></a>

### Accepting Arguments in Decorator Functions
Sometimes, you might need to define a decorator that accepts arguments. You achieve this by **passing the arguments to the wrapper** function. You can then pass these arguments to the function that is being decorated at call time.

In [58]:
def uppercase_decorator(function):
    def wrapper(*args, **kwargs):                    # ← accepts anything
        return function(*args, **kwargs).upper()      # ← passes them through
    return wrapper

@uppercase_decorator
def cities(city_one, city_two):
    return f"I travelled from {city_one} to {city_two}"

cities("Amsterdam", "New York")

'I TRAVELLED FROM AMSTERDAM TO NEW YORK'

In [59]:
def decorator_with_arguments(function):
    
    def wrapper_accepting_arguments(arg1, arg2):
        print(f"My arguments are: {arg1}, {arg2}")
        result = function(arg1, arg2)
        return result
        
    return wrapper_accepting_arguments


@decorator_with_arguments
def cities(city_one, city_two):
    return f"I travelled from {city_one} to {city_two}"

cities("Amsterdam", "New York")

My arguments are: Amsterdam, New York


'I travelled from Amsterdam to New York'

<a id='defining'></a>

### Defining General Purpose Decorators
To define a general purpose decorator that can be applied to any function, you can use **positional arguments** ("args") and **keyword arguments** ("kwargs"). These `*args` and `**kwargs` collect all positional and keyword arguments and store them in the `args` (tuple) and `kwargs` (dictionary) variables. They allow you to pass as many arguments as you would like during function calls.

Let's create `a_decorator_passing_arbitrary_arguments` & `a_wrapper_accepting_arbitrary_arguments`.

In [60]:
def a_decorator_passing_arbitrary_arguments(function):
    
    def a_wrapper_accepting_arbitrary_arguments(*args, **kwargs):
        print('Positional arguments:', args)
        print('Keyword arguments:', kwargs)
        result = function(*args, **kwargs)
        return result
    
    return a_wrapper_accepting_arbitrary_arguments

@a_decorator_passing_arbitrary_arguments
def function_with_no_argument():
    return "No arguments here."

function_with_no_argument()

Positional arguments: ()
Keyword arguments: {}


'No arguments here.'

Let's see how you would use the decorator using positional arguments.

In [61]:
@a_decorator_passing_arbitrary_arguments
def function_with_arguments(a, b, c):
    print(a, b, c)

function_with_arguments(1,2,3)

Positional arguments: (1, 2, 3)
Keyword arguments: {}
1 2 3


Keyword arguments are passed using keywords. An illustration of this is shown below.

In [63]:
def greet(name, age):
    print(f"{name} is {age}")

greet("Daria", 28)              # positional → goes to *args side
greet(name="Daria", age=28)      # keyword → goes to **kwargs side

Daria is 28
Daria is 28


In [62]:
@a_decorator_passing_arbitrary_arguments
def function_with_keyword_arguments(first_name, last_name):
    print(f"This has shown keyword arguments: {first_name} {last_name}")

function_with_keyword_arguments(first_name="Derrick", last_name="Mwiti")

Positional arguments: ()
Keyword arguments: {'first_name': 'Derrick', 'last_name': 'Mwiti'}
This has shown keyword arguments: Derrick Mwiti


### Retaining a function's original documentation

Let's look at this cities example...

In [66]:
def decorator_with_arguments(function):
    
    def wrapper_accepting_arguments(*args, **kwargs):
        """Wrapper function accepting arguments"""
        print('Positional arguments:', args)
        print('Keyword arguments:', kwargs)
        result = function(*args, **kwargs)
        return result
    
    return wrapper_accepting_arguments


@decorator_with_arguments
def cities(city_one, city_two):
    """Print sentence of cities travelled"""
    
    return f"I travelled from {city_one} to {city_two}"

cities(city_one="Amsterdam", city_two="New York")

Positional arguments: ()
Keyword arguments: {'city_one': 'Amsterdam', 'city_two': 'New York'}


'I travelled from Amsterdam to New York'

When you wrap a decorator around a function, the function inherits the documentation from the decorator:

In [67]:
cities.__name__, cities.__doc__

('wrapper_accepting_arguments', 'Wrapper function accepting arguments')

If using a decorator always meant losing this information about a function, it would be a serious problem. Luckily, there is a function called  `functools.wraps` which is itself a decorator. This takes a function used in a decorator and adds the functionality of copying over the function name, docstring, arguments list, etc. 

In [69]:
from functools import wraps

def decorator_with_arguments(function):
    
    @wraps(function)
    def wrapper_accepting_arguments(*args, **kwargs):
        """Wrapper function accepting arguments"""
        print('Positional arguments:', args)
        print('Keyword arguments:', kwargs)
        result = function(*args, **kwargs)
        return result
    
    return wrapper_accepting_arguments


@decorator_with_arguments
def cities(city_one, city_two):
    """Print sentence of cities travelled"""
    return f"I travelled from {city_one} to {city_two}"

cities("Amsterdam", city_two="New York")

Positional arguments: ('Amsterdam',)
Keyword arguments: {'city_two': 'New York'}


'I travelled from Amsterdam to New York'

Now, the `cities` function keeps the original name and docstring!

In [70]:
cities.__name__, cities.__doc__

('cities', 'Print sentence of cities travelled')

<a id='usecases'></a>
## Usecases of decorators

Decorators are widely used within libraries that provide some extensions designed to be used with user-defined functions.

Some common examples include:
* Defining fixtures for testing
* Wraping functions with loggers for monitoring
* Benchmarking functions
* Using Python functions to define external interfaces (e.g. Typer, FastAPI)

### <mark>Exercises</mark>

Here is a function `get_factors` that returns a list of factors for an input number.

*Note: Factors of a number are defined as numbers that divide the original number evenly or exactly. E.g. `[1,2,3,6]` for the number `6`.*

In [71]:
def get_factors(n):
    "Return the factors of n." 
    factors = [x for x in range(1, (n+1))
               if n % x == 0] 
    return factors

get_factors(6)

[1, 2, 3, 6]

★ Create a decorator that prints the positional/keyword arguments and wrap it around `get_factors`.

In [72]:
def decorator_with_arguments(function):
    
    @wraps(function)
    def wrapper_accepting_arguments(*args, **kwargs):
        """Wrapper function accepting arguments"""
        print('Positional arguments:', args)
        print('Keyword arguments:', kwargs)
        result = function(*args, **kwargs)
        return result
    
    return wrapper_accepting_arguments

@decorator_with_arguments
def get_factors(n):
    "Return the factors of n." 
    factors = [x for x in range(1, (n+1))
               if n % x == 0] 
    return factors

get_factors(6)

Positional arguments: (6,)
Keyword arguments: {}


[1, 2, 3, 6]

★★ Create a decorator that prints the name of the function and use `wraps` from functools to retain the name of the function. Check it using `get_factors.__name__`!

In [73]:
def print_names(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        print("Calling function:", function.__name__)
        return function(*args, **kwargs)
    return wrapper
    
    
@print_names
def get_factors(n):
    "Return the factors of n." 
    factors = [x for x in range(1, (n+1))
               if n % x == 0] 
    return factors

get_factors(6)

Calling function: get_factors


[1, 2, 3, 6]

★★★ Create a decorator `is_prime` that also checks whether a number is a prime number or not. Use (and if necessary modify) the `time_it` decorator from before to check how long this takes for bigger numbers. 

In [78]:
import time
from functools import wraps

def time_it(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = function(*args, **kwargs)
        print(f"{function.__name__} took {time.time() - start:.4f}s")
        return result
    return wrapper

def is_prime(function):

    def wrapper(*args):
        result = function(*args)
        if len(result) == 2:
            print(f"{args[0]} is a prime number.")
        else:
            print(f"{args[0]} is not a prime number.")
        return result
    return wrapper
    
    
@time_it
@is_prime
def get_factors(n):
    """Return the factors of n."""
    return [x for x in range(1, n+1) if n % x == 0]

get_factors(7919)        # check a known prime
get_factors(10000000)    # big number → see timing

7919 is a prime number.
wrapper took 0.0005s
10000000 is not a prime number.
wrapper took 0.2101s


[1,
 2,
 4,
 5,
 8,
 10,
 16,
 20,
 25,
 32,
 40,
 50,
 64,
 80,
 100,
 125,
 128,
 160,
 200,
 250,
 320,
 400,
 500,
 625,
 640,
 800,
 1000,
 1250,
 1600,
 2000,
 2500,
 3125,
 3200,
 4000,
 5000,
 6250,
 8000,
 10000,
 12500,
 15625,
 16000,
 20000,
 25000,
 31250,
 40000,
 50000,
 62500,
 78125,
 80000,
 100000,
 125000,
 156250,
 200000,
 250000,
 312500,
 400000,
 500000,
 625000,
 1000000,
 1250000,
 2000000,
 2500000,
 5000000,
 10000000]

**Hint:** A number is prime if it only has two factors: The number itself (`n`) and `1`.

**Answers:**

In [ ]:
# %load answers/ex-decorators2.1.py

def log_function_info(func):

    def wrapper_function(*args, **kwargs):
        print(f"The function name is {func.__name__}")
        print(f"The positional arguments are: {args}")
        print(f"The keyword arguments are: {kwargs}")
        return func(*args, **kwargs)

    return wrapper_function

@log_function_info
def get_factors(n):
    "Return the factors of n."
    factors = [x for x in range(1, (n+1))
               if n % x == 0]
    return factors

get_factors(20)

# get_factors.__name__


In [ ]:
# %load answers/ex-decorators2.2.py
from functools import wraps

def log_function_info(func):

    @wraps(func)
    def wrapper_function(*args, **kwargs):
        print(f"The function name is {func.__name__}")
        print(f"The positional arguments are: {args}")
        print(f"The keyword arguments are: {kwargs}")
        return func(*args, **kwargs)

    return wrapper_function

@log_function_info
def get_factors(n):
    "Return the factors of n."
    factors = [x for x in range(1, (n+1))
               if n % x == 0]
    return factors

get_factors.__name__


In [77]:
# %load answers/ex-decorators2.3.py

def time_it(function):
    import time

    def wrapper(*args):
        t1 = time.time()
        result = function(*args)
        t2 = time.time()
        print(f'The function {function.__name__} took {round(t2-t1, 4)} seconds to run.')
        return result

    return wrapper

def is_prime(function):

    def wrapper(*args):
        result = function(*args)
        if len(result) == 2:
            print(f"{args[0]} is a prime number.")
        else:
            print(f"{args[0]} is not a prime number.")
        return result
    return wrapper

@is_prime
@time_it
def get_factors(n):
    "Return the factors of n."
    factors = [x for x in range(1, (n+1))
               if n % x == 0]
    return factors

get_factors(1254739)


<mark>**Bonus:**</mark> Investigate how to create decorators that accept arguments themselves.

For example, decorator that either lowercases or uppercases the output of another function based on a parameter.

```python 
@change_case(upper = True)
def say_hi():
    return "Hello!"

say_hi("people")
>>> "HELLO PEOPLE!"
```

In [ ]:
# %load answers/ex-decorators2-bonus.py
from functools import wraps


def change_case(upper):

    def decorator(function):

        def wrapper(*args, **kwargs):
            if upper:
                return function(*args, **kwargs).upper()
            return function(*args, **kwargs)

        return wrapper

    return decorator

@change_case(upper=True)
def say_hi(who):
    return f"Hello {who}!"

say_hi("people")
